## tl;dr

O salto visual de V5 (67,06%) para o candidato (77,10%) não representa ganho textual comparável: 99,9916% do salto vem da troca do modelo #358 pelo modelo ativo #76. No par comparável #379 → #380, 99 textos mudaram, todos receberam score maior, mas o efeito global foi de apenas +0,0008478 p.p. A calibração atual está pendente e adicionou zero ao índice.

## Context & Methods

Diagnóstico do índice global e das faixas exibidas no dashboard em 19 de julho de 2026. A unidade é o segmento ativo; a fonte canônica é `memory/translation_engine.sqlite`.

### Key Assumptions

- Scores só são comparados diretamente quando regra, modelo e snapshot coincidem.
- O índice global é a média aritmética de `model_safe_probability` nos 287.895 segmentos medidos.
- Score do modelo é evidência de qualidade, não substitui validação humana.

## Data

In [ ]:
import json
import sqlite3
from pathlib import Path

project_root = Path.cwd().resolve()
db_path = project_root / 'memory' / 'translation_engine.sqlite'
connection = sqlite3.connect(f'file:{db_path.as_posix()}?mode=ro', uri=True)
connection.row_factory = sqlite3.Row

def query(sql, params=()):
    return [dict(row) for row in connection.execute(sql, params)]

assert db_path.exists(), db_path
print(db_path)

## Results

### 1. Comparar os runs sob o mesmo contrato

In [ ]:
score_runs = query('''
SELECT id, rule_version, model_run_id, model_version, source_snapshot_id,
       candidate_text_source, candidate_tree_hash, scored_count, finished_at
FROM ml_score_runs WHERE id IN (379, 380) ORDER BY id
''')
score_aggregate = query('''
SELECT run_id, COUNT(*) AS rows, COUNT(DISTINCT segment_id) AS distinct_segments,
       AVG(model_safe_probability) AS avg_score,
       SUM(CASE WHEN model_safe_probability < .20 THEN 1 ELSE 0 END) AS critical,
       SUM(CASE WHEN model_safe_probability >= .20 AND model_safe_probability < .50 THEN 1 ELSE 0 END) AS low,
       SUM(CASE WHEN model_safe_probability >= .50 AND model_safe_probability < .75 THEN 1 ELSE 0 END) AS moderate,
       SUM(CASE WHEN model_safe_probability >= .75 AND model_safe_probability < .90 THEN 1 ELSE 0 END) AS good,
       SUM(CASE WHEN model_safe_probability >= .90 THEN 1 ELSE 0 END) AS high
FROM ml_score_items WHERE run_id IN (379, 380) GROUP BY run_id ORDER BY run_id
''')
print(json.dumps(score_runs, ensure_ascii=False, indent=2))
print(json.dumps(score_aggregate, ensure_ascii=False, indent=2))

### 2. Isolar o efeito dos 99 textos alterados

In [ ]:
text_decomposition = query('''
WITH pairs AS (
 SELECT a.segment_id, s.old_text, o.portuguese_text,
        a.model_safe_probability AS old_score,
        b.model_safe_probability AS new_score,
        b.model_safe_probability-a.model_safe_probability AS delta
 FROM ml_score_items a
 JOIN ml_score_items b ON b.segment_id=a.segment_id AND b.run_id=380
 JOIN source_segments s ON s.id=a.segment_id AND s.is_active=1
 LEFT JOIN output_segments o ON o.segment_id=a.segment_id
 WHERE a.run_id=379
)
SELECT CASE WHEN old_text IS portuguese_text THEN 'text_unchanged' ELSE 'text_changed' END AS cohort,
       COUNT(*) AS segments, AVG(old_score) AS avg_old, AVG(new_score) AS avg_new,
       AVG(delta) AS avg_delta, SUM(delta) AS delta_sum,
       SUM(CASE WHEN delta > .0001 THEN 1 ELSE 0 END) AS improved,
       SUM(CASE WHEN delta < -.0001 THEN 1 ELSE 0 END) AS regressed,
       SUM(CASE WHEN ABS(delta) <= .0001 THEN 1 ELSE 0 END) AS equal
FROM pairs GROUP BY cohort ORDER BY cohort
''')
print(json.dumps(text_decomposition, ensure_ascii=False, indent=2))

### 3. Decompor V5 → candidato entre troca de modelo e mudança textual

In [ ]:
version_model = query('''
SELECT v.version_number, v.full_average_score, v.score_run_id,
       r.model_run_id, r.model_version, r.candidate_tree_hash
FROM package_versions v
LEFT JOIN ml_score_runs r ON r.id=v.score_run_id
WHERE v.version_number=5
''')[0]
v5_score = version_model['full_average_score']
v5_rescored_current_model = score_aggregate[0]['avg_score']
current_output_score = score_aggregate[1]['avg_score']
model_effect_pp = (v5_rescored_current_model-v5_score)*100
text_effect_pp = (current_output_score-v5_rescored_current_model)*100
total_jump_pp = (current_output_score-v5_score)*100
decomposition = {
    'v5_score_pct': v5_score*100,
    'v5_rescored_model_76_pct': v5_rescored_current_model*100,
    'current_output_pct': current_output_score*100,
    'model_effect_pp': model_effect_pp,
    'text_effect_pp': text_effect_pp,
    'total_jump_pp': total_jump_pp,
    'model_share_pct': model_effect_pp/total_jump_pp*100,
    'text_share_pct': text_effect_pp/total_jump_pp*100,
}
print(json.dumps(decomposition, ensure_ascii=False, indent=2))

### 4. Verificar se a calibração alterou o índice

In [ ]:
calibration = query('''
SELECT id, quality_epoch_id, old_score_run_id, output_score_run_id,
       pending_count, decided_count, consumption_status, consumed_count, control_accuracy
FROM ml_pairwise_calibration_review_runs WHERE id IN (6, 7) ORDER BY id
''')
prior_reasons = query('''
SELECT reviewer_reason, COUNT(*) AS segments, AVG(raw_delta) AS avg_raw_delta
FROM ml_pairwise_calibration_review_items WHERE run_id=6
GROUP BY reviewer_reason ORDER BY segments DESC
''')
print(json.dumps(calibration, ensure_ascii=False, indent=2))
print(json.dumps(prior_reasons, ensure_ascii=False, indent=2))

## Takeaways

- O ganho textual comparável existe no score do modelo ativo, mas é minúsculo no pacote inteiro: +0,0008478 p.p.
- A subida de aproximadamente 10,0443 p.p. no gráfico histórico mistura modelos diferentes; cerca de 10,0434 p.p. vêm da troca #358 → #76.
- A barra Alta não cresceu: ela ficou em 134.820 nos runs #379 e #380. Dez segmentos migraram de Crítico para Baixo, sem alterar o total abaixo de 50%.
- A calibração anterior validou 95 correções `d` → `de`; ela aumentou a confiança sem reescrever o score bruto. A calibração atual tem 16 controles pendentes e efeito zero no índice.
- O dashboard deve separar evolução textual sob contrato fixo de mudança de régua/modelo.